Reference:
1. https://platform.openai.com/docs/guides/fine-tuning/when-to-use-fine-tuning
2. https://github.com/norahsakal/fine-tune-gpt3-model/blob/main/fine_tune_step_by_step.ipynb
3. Course: https://app.gumroad.com/d/755cba85b168cd239e458e0cad53243e
4. https://norahsakal.com/blog/fine-tune-gpt3-model

Example format

In this example, our goal is to create a chatbot that occasionally gives sarcastic responses, these are three training examples (conversations) we could create for a dataset:

```json
{"messages": [{"role": "system", "content": "Marv is a factual chatbot that is also sarcastic."}, {"role": "user", "content": "What's the capital of France?"}, {"role": "assistant", "content": "Paris, as if everyone doesn't know that already."}]}
{"messages": [{"role": "system", "content": "Marv is a factual chatbot that is also sarcastic."}, {"role": "user", "content": "Who wrote 'Romeo and Juliet'?"}, {"role": "assistant", "content": "Oh, just some guy named William Shakespeare. Ever heard of him?"}]}
{"messages": [{"role": "system", "content": "Marv is a factual chatbot that is also sarcastic."}, {"role": "user", "content": "How far is the Moon from Earth?"}, {"role": "assistant", "content": "Around 384,400 kilometers. Give or take a few, like that really matters."}]}
```


The conversational chat format is required to fine-tune gpt-3.5-turbo. For babbage-002 and davinci-002, you can follow the prompt completion pair format used for legacy fine-tuning as shown below.

```json
{"prompt": "<prompt text>", "completion": "<ideal generated text>"}
{"prompt": "<prompt text>", "completion": "<ideal generated text>"}
{"prompt": "<prompt text>", "completion": "<ideal generated text>"}
```

In [ ]:
# Fine tune the gpt-3 model
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

In [ ]:
# Create training data

In [ ]:
training_data = [
    {
        "prompt": "Where is the billing ->",
        "completion": " You find the billing in the left-hand side menu.\n",
    },
    {
        "prompt": "How do I upgrade my account ->",
        "completion": " Visit you user settings in the left-hand side menu, then click 'upgrade account' button at the top.\n",
    },
]

Make sure to end each prompt with a suffix. According to the OpenAI API reference, you can use ->.

Also, make sure to end each completion with a suffix as well; I'm using .\n.

The next step is to convert the dict to a proper JSONL file. JSONL file is a newline-delimited JSON file, so we'll add a \n at the end of each object:

In [ ]:
import json

In [ ]:
file_name = 'training_data_openai.jsonl'

with open(file_name, 'w') as file:
    for entry in training_data:
        json.dump(entry, file)
        file.write('\n')

In [ ]:
# Check the training data

In [ ]:
import openai

In [ ]:
!openai tools fine_tunes.prepare_data -f training_data_openai.jsonl

In [ ]:
# Upload training data

In [ ]:
upload_response = openai.File.create(
    file=open(file_name, 'rb'),
    purpose='fine-tune'
)
file_id = upload_response.id
upload_response

If you check the response, you'll see the file id which we'll need in the next step when we're training the model. Use this file id in the next step, where we'll fine-tune a model.

In [ ]:
# Fine-tune model

In [ ]:
fine_tune_response = openai.FineTune.create(training_file=file_id) # mention the id of file in the training file to train the model based on our data
fine_tune_response

The default model is Curie. But if you'd like to use DaVinci instead, then add it as a base model to fine-tune like this:

```
openai.FineTune.create(training_file=file_id, model="davinci")
```

In [ ]:
# Check fine-tuning progress

In [ ]:
fine_tune_events = openai.FineTune.list_events(id=fine_tune_response.id)
fine_tune_events

In [ ]:
# Retrieve fine-tuning job

In [ ]:
retrieve_response = openai.FineTune.retrieve(id=fine_tune_response.id)
retrieve_response

In [ ]:
# Save fine-tuned model

In [ ]:
if fine_tune_response.fine_tuned_model != None:
    fine_tuned_model = fine_tune_response.fine_tuned_model
    fine_tuned_model
else:
    retrieve_response = openai.FineTune.retrieve(fine_tune_response.id)
    fine_tuned_model = retrieve_response.fine_tuned_model

fine_tuned_model

In [ ]:
# Test the new model on a new prompt

In [ ]:
# training_data = [
#     {
#         "prompt": "Where is the billing ->",
#         "completion": " You find the billing in the left-hand side menu.\n",
#     },
#     {
#         "prompt": "How do I upgrade my account ->",
#         "completion": " Visit you user settings in the left-hand side menu, then click 'upgrade account' button at the top.\n",
#     },
# ]

If you will observe then there is no such prompt called "How do I find my billing? ->", but it's answer should be similar to the answer of the prompt "Where is the billing ->". So, let's give it to fine tuned model and see the output

In [ ]:
new_prompt = "How do I find my billing? ->"

In [ ]:
answer = openai.Completion.create(
    model=fine_tuned_model,
    prompt=new_prompt,
    max_tokens=10,
    temperature=0
)

In [ ]:
print(answer['choices'][0]['text'])

In [ ]:
new_prompt2 = "Is there any way to upgrade my account? ->"

In [ ]:
answer = openai.Completion.create(
    model=fine_tuned_model,
    prompt=new_prompt2,
    max_tokens=100,
    temperature=0
)

In [ ]:
print(answer['choices'][0]['text'])

As observe, the same answer is there multiple times in the response, due to high number of tokens. How to deal with this?

# Now let's train the gpt-3.5-turbo model

In [ ]:
import openai

In [ ]:
file_id = "file-RBFqP5H2iwG1rtyZR2BFlvbx"

In [ ]:
# Note: The given code gives the error that gpt-3.5-turbo 
# can only be fine-tuned on the new fine-tuning API (`/fine_tuning/jobs`). This API (`/fine-tunes`) is being deprecated. Please refer to our documentation for more information: https://platform.openai.com/docs/api-reference/fine-tuning 
fine_tune_response = openai.FineTune.create(training_file=file_id, model='gpt-3.5-turbo')
fine_tune_response

In [ ]:
%pip install --upgrade openai 

In [ ]:
fine_tune_response = openai.FineTuningJob.create(training_file=file_id, model='gpt-3.5-turbo')

In [ ]:
# Observe this the previous model required the training in prompt-completion format, 
# but this model requires in chat completion format. So let's go and train the gpt-3-turbo

# Refer the fine_tuning_gpt-3.5-turbo.ipynb